In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🏥 Medical Literature Finder

## What We're Building

A RAG system that retrieves medical studies and filters them by:
- **Condition** (e.g., Type 2 Diabetes, Hypertension)
- **Outcome** (e.g., HbA1c reduction, blood pressure)
- **Evidence Strength** (RCT, meta-analysis, cohort study, case report)

## Why Metadata Filters Matter in Medical RAG

In medicine, not all evidence is equal. A meta-analysis of 50 RCTs is
stronger than a single case report. By tagging each study with its
**evidence level** and **study type**, we can:

- Retrieve only high-quality evidence (RCTs + meta-analyses)
- Filter by condition to avoid irrelevant results
- Sort by recency for the latest evidence

## Evidence Hierarchy (simplified)

```
Level 1: Systematic Reviews / Meta-Analyses  (strongest)
Level 2: Randomized Controlled Trials (RCTs)
Level 3: Cohort Studies
Level 4: Case-Control Studies
Level 5: Case Reports / Expert Opinion        (weakest)
```

## Stack
- **LangChain** — orchestration
- **ChromaDB** — vector store with metadata filtering
- **Ollama** — local LLM + embeddings

## Step 1 — Imports

In [ ]:
# !pip install langchain langchain-ollama langchain-community chromadb -q

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Simulated Medical Studies

Each study has structured metadata for filtering. In production, you'd
parse this from PubMed XML or clinical trial registries.

In [ ]:
studies = [
    {
        "title": "SUSTAIN-6: Semaglutide and Cardiovascular Outcomes in Type 2 Diabetes",
        "authors": "Marso SP, Bain SC, et al.",
        "year": 2016,
        "journal": "NEJM",
        "study_type": "RCT",
        "evidence_level": 2,
        "condition": "type_2_diabetes",
        "outcome": "cardiovascular",
        "sample_size": 3297,
        "abstract": """Background: Semaglutide is a glucagon-like peptide-1 (GLP-1) analogue.
We assessed its cardiovascular safety in patients with type 2 diabetes.

Methods: We randomized 3,297 patients with type 2 diabetes and high cardiovascular
risk to receive semaglutide (0.5 mg or 1.0 mg) or placebo for 104 weeks.

Results: The primary composite outcome (CV death, nonfatal MI, nonfatal stroke)
occurred in 6.6% of semaglutide vs 8.9% of placebo (HR 0.74, 95% CI 0.58-0.95,
p<0.001 for noninferiority). Nonfatal stroke was reduced by 39% (HR 0.61).
HbA1c was reduced by 1.0-1.4% from baseline. Body weight decreased by 3.6-4.9 kg.

Conclusions: Semaglutide significantly reduced cardiovascular events compared
to placebo in type 2 diabetes patients with high CV risk.""",
    },
    {
        "title": "EMPA-REG OUTCOME: Empagliflozin and Cardiovascular Outcomes in Type 2 Diabetes",
        "authors": "Zinman B, Wanner C, et al.",
        "year": 2015,
        "journal": "NEJM",
        "study_type": "RCT",
        "evidence_level": 2,
        "condition": "type_2_diabetes",
        "outcome": "cardiovascular",
        "sample_size": 7020,
        "abstract": """Background: Empagliflozin is an SGLT2 inhibitor approved for type 2 diabetes.
Its cardiovascular effects were unknown.

Methods: We randomized 7,020 patients to empagliflozin (10 mg or 25 mg) or
placebo. Median follow-up was 3.1 years.

Results: The primary outcome (CV death, nonfatal MI, nonfatal stroke) occurred
in 10.5% of empagliflozin vs 12.1% of placebo (HR 0.86, 95% CI 0.74-0.99,
p=0.04). Cardiovascular death was reduced by 38% (HR 0.62). Heart failure
hospitalization was reduced by 35% (HR 0.65). HbA1c reduction was 0.3-0.4%.

Conclusions: Empagliflozin reduced the risk of cardiovascular death and heart
failure hospitalization in patients with type 2 diabetes and established CV disease.""",
    },
    {
        "title": "GLP-1 Receptor Agonists for Weight Management: A Systematic Review",
        "authors": "Smith A, Johnson B, et al.",
        "year": 2023,
        "journal": "Lancet Diabetes Endocrinol",
        "study_type": "systematic_review",
        "evidence_level": 1,
        "condition": "obesity",
        "outcome": "weight_loss",
        "sample_size": 22000,
        "abstract": """Background: GLP-1 receptor agonists have shown promise for weight management
beyond glycemic control.

Methods: We systematically reviewed 24 RCTs (n=22,000) of GLP-1 RA for weight
loss in adults with BMI ≥30 or BMI ≥27 with comorbidities.

Results: Semaglutide 2.4 mg weekly produced the largest weight reduction:
-15.3% body weight at 68 weeks (vs -2.6% placebo). Liraglutide 3.0 mg produced
-7.5% weight loss. Tirzepatide 15 mg produced -20.9% weight loss at 72 weeks.
Common side effects were nausea (35-45%), diarrhea (20-30%), and vomiting (15-25%).
Discontinuation due to GI side effects was 5-8%.

Conclusions: GLP-1 RAs produce clinically significant weight loss. Tirzepatide
and high-dose semaglutide show the greatest efficacy. GI side effects are common
but generally manageable.""",
    },
    {
        "title": "SPRINT Trial: Intensive vs Standard Blood Pressure Control",
        "authors": "SPRINT Research Group",
        "year": 2015,
        "journal": "NEJM",
        "study_type": "RCT",
        "evidence_level": 2,
        "condition": "hypertension",
        "outcome": "blood_pressure",
        "sample_size": 9361,
        "abstract": """Background: The optimal systolic blood pressure target for patients at high
cardiovascular risk is debated.

Methods: We randomized 9,361 non-diabetic adults with SBP ≥130 and high CV risk
to intensive treatment (target SBP <120) or standard treatment (target SBP <140).

Results: The trial was stopped early due to benefit. The primary composite outcome
(MI, ACS, stroke, HF, or CV death) occurred at 1.65%/year (intensive) vs
2.19%/year (standard), HR 0.75, p<0.001. All-cause mortality was also reduced.
However, serious adverse events including hypotension (2.4% vs 1.4%), syncope
(2.3% vs 1.7%), and acute kidney injury (4.1% vs 2.5%) were more common.

Conclusions: Intensive blood pressure control (SBP <120) significantly reduced
cardiovascular events and mortality compared to standard control (<140) in high-risk
non-diabetic adults, but increased adverse events.""",
    },
    {
        "title": "Metformin as First-Line Therapy for Type 2 Diabetes: A Meta-Analysis",
        "authors": "Liu R, Zhang K, et al.",
        "year": 2022,
        "journal": "Diabetes Care",
        "study_type": "meta_analysis",
        "evidence_level": 1,
        "condition": "type_2_diabetes",
        "outcome": "glycemic_control",
        "sample_size": 45000,
        "abstract": """Background: Metformin has been the cornerstone first-line therapy for T2D
for decades, but newer agents challenge this position.

Methods: Meta-analysis of 38 RCTs (n=45,000) comparing metformin monotherapy
to other first-line options (SGLT2i, GLP-1 RA, DPP-4i, sulfonylureas).

Results: Metformin reduced HbA1c by 1.1% (95% CI 0.9-1.3%) from baseline.
SGLT2 inhibitors showed similar HbA1c reduction (1.0%) but superior weight loss
(-2.5 kg vs -0.6 kg) and cardiovascular outcomes. GLP-1 RAs showed greater
HbA1c reduction (1.3%) and weight loss (-3.2 kg). Metformin had the best
cost-effectiveness ratio and the longest safety track record (>60 years).
GI side effects (diarrhea, nausea) occurred in 25% of metformin users.

Conclusions: Metformin remains a reasonable first-line choice due to efficacy,
safety profile, and cost. However, for patients with established CVD or high
CV risk, SGLT2i or GLP-1 RA should be preferred regardless of HbA1c.""",
    },
    {
        "title": "Hypertension in CKD: KDIGO 2024 Clinical Practice Guideline",
        "authors": "Kidney Disease: Improving Global Outcomes",
        "year": 2024,
        "journal": "Kidney International",
        "study_type": "clinical_guideline",
        "evidence_level": 1,
        "condition": "hypertension",
        "outcome": "kidney_outcomes",
        "sample_size": 0,
        "abstract": """This clinical practice guideline addresses blood pressure management in
patients with chronic kidney disease (CKD).

Key Recommendations:
1. Target SBP <120 mmHg for CKD patients (Grade 1B recommendation)
2. ACE inhibitors or ARBs are first-line for CKD with proteinuria (Grade 1A)
3. SGLT2 inhibitors recommended for CKD patients with eGFR ≥20 (Grade 1A)
4. Avoid NSAIDs in CKD patients with hypertension
5. Monitor potassium when using RAASi — discontinue if K >5.5 mmol/L
6. Dihydropyridine CCBs (amlodipine) preferred as add-on therapy
7. Home blood pressure monitoring recommended for all CKD patients

Evidence basis: Recommendations derived from SPRINT CKD subgroup (n=2,646),
CREDENCE trial, DAPA-CKD trial, and EMPA-KIDNEY trial. Strong evidence supports
intensive BP control in CKD for reducing cardiovascular and renal events.""",
    },
    {
        "title": "Case Report: Metformin-Associated Lactic Acidosis in Renal Impairment",
        "authors": "Chen J, Williams M",
        "year": 2023,
        "journal": "Case Reports in Medicine",
        "study_type": "case_report",
        "evidence_level": 5,
        "condition": "type_2_diabetes",
        "outcome": "adverse_event",
        "sample_size": 1,
        "abstract": """We report a case of a 72-year-old male with type 2 diabetes and stage 3b CKD
who developed severe lactic acidosis (pH 6.9, lactate 18 mmol/L) while on
metformin 2000 mg/day. The patient presented with altered consciousness and
was treated with bicarbonate infusion and hemodialysis.

Contributing factors: acute kidney injury from dehydration during a febrile
illness, which reduced metformin clearance. The patient's eGFR had declined
from 38 to 12 mL/min during the illness.

This case reinforces the need for metformin dose adjustment in CKD and
temporary discontinuation during acute illness (sick day rules). Current
guidelines recommend eGFR ≥30 for metformin use with dose reduction at
eGFR 30-45.""",
    },
    {
        "title": "Physical Activity and Hypertension: A Cohort Study of 50,000 Adults",
        "authors": "Anderson P, Lee S, et al.",
        "year": 2023,
        "journal": "JAMA Internal Medicine",
        "study_type": "cohort_study",
        "evidence_level": 3,
        "condition": "hypertension",
        "outcome": "blood_pressure",
        "sample_size": 50000,
        "abstract": """Background: The dose-response relationship between physical activity and
blood pressure reduction is not well quantified.

Methods: Prospective cohort study of 50,000 adults followed for 5 years with
accelerometer-measured physical activity and serial blood pressure measurements.

Results: Compared to sedentary adults (<2,000 steps/day), those achieving:
- 5,000-7,500 steps/day had 2.3 mmHg lower SBP (95% CI 1.8-2.8)
- 7,500-10,000 steps/day had 4.1 mmHg lower SBP (95% CI 3.4-4.8)
- >10,000 steps/day had 5.2 mmHg lower SBP (95% CI 4.3-6.1)

150 minutes/week of moderate-intensity exercise was associated with a 30%
reduction in incident hypertension (HR 0.70, 95% CI 0.63-0.78).

Conclusions: Physical activity shows a dose-response relationship with blood
pressure reduction. Even modest increases in daily steps provide measurable
benefit.""",
    },
]

print(f"Loaded {len(studies)} medical studies:")
for s in studies:
    print(f"  [{s['study_type']:18s}] (L{s['evidence_level']}) {s['title'][:60]}...")

## Step 3 — Index Studies with Metadata for Filtering

In [ ]:
docs = []
for study in studies:
    doc = Document(
        page_content=f"Title: {study['title']}\n\n{study['abstract']}",
        metadata={
            "title": study["title"],
            "authors": study["authors"],
            "year": study["year"],
            "journal": study["journal"],
            "study_type": study["study_type"],
            "evidence_level": study["evidence_level"],
            "condition": study["condition"],
            "outcome": study["outcome"],
            "sample_size": study["sample_size"],
        },
    )
    docs.append(doc)

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="medical_literature",
)

print(f"Indexed {vectorstore._collection.count()} studies")

## Step 4 — Filtered Search Functions

ChromaDB's `$where` filter lets us search with metadata constraints —
e.g., "only RCTs about diabetes".

In [ ]:
EVIDENCE_LABELS = {
    1: "Level 1 (Systematic Review / Meta-Analysis / Guideline)",
    2: "Level 2 (Randomized Controlled Trial)",
    3: "Level 3 (Cohort Study)",
    4: "Level 4 (Case-Control Study)",
    5: "Level 5 (Case Report / Expert Opinion)",
}


def search_literature(
    query: str,
    condition: str | None = None,
    max_evidence_level: int | None = None,
    study_type: str | None = None,
    k: int = 4,
) -> list:
    """Search medical literature with optional metadata filters."""
    # Build ChromaDB where filter
    filters = []
    if condition:
        filters.append({"condition": condition})
    if max_evidence_level:
        filters.append({"evidence_level": {"$lte": max_evidence_level}})
    if study_type:
        filters.append({"study_type": study_type})
    
    where_filter = None
    if len(filters) == 1:
        where_filter = filters[0]
    elif len(filters) > 1:
        where_filter = {"$and": filters}
    
    # Search with filters
    if where_filter:
        results = vectorstore.similarity_search(
            query, k=k, filter=where_filter
        )
    else:
        results = vectorstore.similarity_search(query, k=k)
    
    return results


def display_results(results: list) -> None:
    """Pretty-print search results with evidence level badges."""
    for i, doc in enumerate(results, 1):
        m = doc.metadata
        level_label = EVIDENCE_LABELS.get(m["evidence_level"], "Unknown")
        print(f"\n{'='*60}")
        print(f"Result {i}: {m['title']}")
        print(f"  Authors: {m['authors']}")
        print(f"  Journal: {m['journal']} ({m['year']})")
        print(f"  Study Type: {m['study_type']}")
        print(f"  Evidence: {level_label}")
        print(f"  Condition: {m['condition']} | Outcome: {m['outcome']}")
        if m['sample_size'] > 0:
            print(f"  Sample Size: n={m['sample_size']:,}")
        print(f"  ---")
        # Show first 200 chars of abstract
        abstract_preview = doc.page_content.split("\n\n", 1)[-1][:200]
        print(f"  {abstract_preview}...")


print("Literature search ready!")

## Step 5 — Search with Filters

In [ ]:
# Search 1: All diabetes studies
print("🔍 Search: Diabetes treatments — ALL evidence levels")
results = search_literature("diabetes treatment outcomes", condition="type_2_diabetes")
display_results(results)

In [ ]:
# Search 2: Only high-quality evidence (Level 1-2)
print("🔍 Search: Diabetes cardiovascular outcomes — HIGH evidence only (L1-L2)")
results = search_literature(
    "cardiovascular outcomes in diabetes",
    condition="type_2_diabetes",
    max_evidence_level=2,
)
display_results(results)

In [ ]:
# Search 3: Hypertension treatment
print("🔍 Search: Blood pressure management")
results = search_literature("blood pressure control treatment", condition="hypertension")
display_results(results)

In [ ]:
# Search 4: No filters — cross-condition search
print("🔍 Search: Weight loss medications — ALL conditions")
results = search_literature("weight loss medication", k=3)
display_results(results)

## Step 6 — LLM-Powered Evidence Summary

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

evidence_prompt = ChatPromptTemplate.from_template(
    """You are a medical literature assistant. Summarize the retrieved studies
to answer the clinical question.

GUIDELINES:
- Cite specific studies by author and year
- Note the evidence level for each study (Level 1 = strongest, Level 5 = weakest)
- Highlight effect sizes and confidence intervals when available
- Note any conflicts or gaps in the evidence
- Include sample sizes to contextualize findings
- End with a brief evidence quality assessment

DISCLAIMER: This is for educational purposes only. Always consult medical professionals.

Retrieved Studies:
{context}

Clinical Question: {question}

Evidence Summary:"""
)


def ask_medical(question: str, condition: str | None = None, max_level: int | None = None) -> None:
    """Ask a clinical question with optional filters."""
    print(f"🏥 Question: {question}")
    if condition:
        print(f"   Filter: condition={condition}")
    if max_level:
        print(f"   Filter: max evidence level={max_level}")
    print("=" * 60)
    
    results = search_literature(question, condition=condition, max_evidence_level=max_level)
    
    context = "\n\n---\n\n".join(
        f"Study: {d.metadata['title']}\n"
        f"Authors: {d.metadata['authors']} ({d.metadata['year']})\n"
        f"Journal: {d.metadata['journal']}\n"
        f"Evidence Level: {d.metadata['evidence_level']} ({EVIDENCE_LABELS[d.metadata['evidence_level']]})\n"
        f"Sample Size: {d.metadata['sample_size']}\n\n"
        f"{d.page_content}"
        for d in results
    )
    
    chain = evidence_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    
    print(f"\n{answer}")
    print("\n" + "=" * 60)


print("Medical evidence assistant ready!")

In [ ]:
ask_medical(
    "What is the best first-line medication for type 2 diabetes with cardiovascular risk?",
    condition="type_2_diabetes",
    max_level=2,
)

In [ ]:
ask_medical(
    "What is the optimal blood pressure target and what treatments are recommended?",
    condition="hypertension",
)

In [ ]:
ask_medical(
    "What are the risks of using metformin in patients with kidney disease?"
)

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Evidence levels** | Tag each study with its position in the evidence hierarchy |
| **ChromaDB metadata filters** | `$lte`, `$and` filters for condition, level, type |
| **Structured study metadata** | Authors, year, journal, sample size — all searchable |
| **Evidence quality assessment** | LLM evaluates strength of the retrieved evidence |
| **Cross-condition search** | Remove filters to search across all conditions |

## 🔧 Customization Ideas

- **PubMed integration**: Fetch real abstracts via the PubMed API
- **PICO extraction**: Parse Population, Intervention, Comparison, Outcome
- **Citation network**: Show which studies reference each other
- **Recency weighting**: Boost more recent studies in retrieval